<a href="https://colab.research.google.com/github/protein2005/neuralnetworks/blob/master/mkr/notebookb6d20c6c08.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install ultralytics roboflow albumentations opencv-python-headless

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 44.7 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.12.0.88
    Uninstalling opencv-python-headless-4.12.0.88:
      Successfully uninstalled opencv-python-headless-4.12.0.88
  Attempting uninstall: idna
    Found existing installation: idna 3.11
    Uninstalling idna-3.11:
      Successfully uninstalled idna-3.11


In [2]:
import os
from google.colab import files

print("Будь ласка, завантажте файл kaggle.json:")
uploaded = files.upload()
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

print("\nЗавантаження датасету oleksiisokolchuk/mkr-data...")
!kaggle datasets download -d oleksiisokolchuk/mkr-data
dataset_dir = "/content/dataset"
if os.path.exists(dataset_dir):
    import shutil
    shutil.rmtree(dataset_dir)
os.makedirs(dataset_dir, exist_ok=True)

!unzip -q mkr-data.zip -d {dataset_dir}
print("Готово!")


import yaml
import glob

yaml_files = glob.glob(f"{dataset_dir}/**/data.yaml", recursive=True)

if yaml_files:
    yaml_path = yaml_files[0]
    base_path = os.path.dirname(yaml_path)

    with open(yaml_path, 'r') as f:
        data = yaml.safe_load(f)
    data['path'] = base_path
    data['train'] = "train/images"
    data['val'] = "valid/images"
    if 'test' in data:
        data['test'] = "test/images"

    with open(yaml_path, 'w') as f:
        yaml.dump(data, f)

    print("Файл data.yaml успішно оновлено для роботи в Colab!")
    print(f"Класи: {data['names']}")
else:
    print("ПОМИЛКА: Не знайдено файл data.yaml у завантаженому архіві!")

import albumentations as A
import cv2
import matplotlib.pyplot as plt
import random

train_images_dir = os.path.join(base_path, "train", "images")
image_files = os.listdir(train_images_dir)

if image_files:
    rand_img = random.choice(image_files)
    img_path = os.path.join(train_images_dir, rand_img)

    image = cv2.imread(img_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    # 7 методів аугментації
    transform = A.Compose([
        A.HorizontalFlip(p=0.5),
        A.RandomBrightnessContrast(p=0.2),
        A.Rotate(limit=20, p=0.5),
        A.GaussianBlur(p=0.2),
        A.RGBShift(p=0.2),
        A.RandomFog(p=0.1),
        A.CLAHE(p=0.5),
        A.ChannelShuffle(p=0.1)
    ])

else:
    print("Папка з зображеннями порожня!")

Будь ласка, завантажте файл kaggle.json:


Saving kaggle.json to kaggle.json

Завантаження датасету oleksiisokolchuk/mkr-data...
Dataset URL: https://www.kaggle.com/datasets/oleksiisokolchuk/mkr-data
License(s): unknown
  0% 0.00/7.32M [00:00<?, ?B/s]
100% 7.32M/7.32M [00:00<00:00, 1.17GB/s]
Готово!
Файл data.yaml успішно оновлено для роботи в Colab!
Класи: ['toyota-logo']


In [3]:
from ultralytics import YOLO

model = YOLO('yolo11n.pt')

results = model.train(
    data='/content/dataset/data.yaml',
    epochs=100,           #
    patience=0,
    batch=8,
    imgsz=640,
    name='mkr_neiro',
    optimizer='AdamW',
    lr0=0.001,
    exist_ok=True
)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.3.233 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=Fals

In [4]:
from google.colab import files
import os

uploaded = files.upload()
video_path = list(uploaded.keys())[0]

best_weights = "/content/runs/detect/mkr_neiro/weights/best.pt"

if os.path.exists(best_weights):
    print(f"Завантаження моделі: {best_weights}")
    model = YOLO(best_weights)

    results = model.predict(source=video_path, save=True, conf=0.69)

    print("\nВідео оброблено успішно!")

    import glob
    latest_predict = sorted(glob.glob('/content/runs/detect/predict*'))[-1]
    print(f"Результат збережено в: {latest_predict}")

else:
    print("Помилка: Файл ваг best.pt не знайдено. Навчання не завершилося успішно.")

Saving Toyota.mp4 to Toyota.mp4
Завантаження моделі: /content/runs/detect/mkr_neiro/weights/best.pt

WARNING ⚠️ 
inference results will accumulate in RAM unless `stream=True` is passed, causing potential out-of-memory
errors for large sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/2716) /content/Toyota.mp4: 384x640 (no detections), 54.5ms
video 1/1 (frame 2/2716) /content/Toyota.mp4: 384x640 (no detections), 9.0ms
video 1/1 (frame 3/2716) /content/Toyota.mp4: 384x640 1 toyota-logo, 16.9ms
video 1/1 (frame 4/2716) /content/Toyota.mp4: 384x640 1 toyota-logo, 14.2ms
video 1/1 (frame 5/2716) /content/Toyota.mp4: 384